In [9]:
import pltkit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.stats import qmc
import sys, os, glob, re
sys.path.append(os.path.abspath(".."))    
import models.Carleton2022.model.mortality_functions as mf

#### Exposure Response Functions

In [ ]:
wdir = "X:/user/liprandicn/mt-comparison/carleton2022"

sets = mf.ModelSettings(
        temp_dir="X:/user/liprandicn/Data/ERA5/t2m_daily",
        gdp_dir=None,
        wdir=wdir,
        project="default",
        scenario="SSP2_ERA5",
        years=range(1980, 2015),
        adaptation=None,
        counterfactual=None,
        reporting_tool=None,
        draw="mean"
    )
fls = mf.LoadInputData.from_files(sets=sets)

erfs_t0,_ = mf.GenerateERFAll(
            sets=sets, 
            fls=fls,
            year=None, 
            adaptation=False, 
            baseline=None,
            counterfactual=None,
            ) 


fig, axs = plt.subplots(1,3, figsize=(15,5), dpi=300)
axs = axs.flatten()
t = np.arange(-20, 40.1, 0.1).round(1)
age_groups = ["+65 years", "5-64 years", "0-4 years"]

for i, age in enumerate(["oldest", "older", "young"]):
    axs[i].plot(t, erfs_t0[age].mean(axis=0), color="k", linewidth=2, zorder=3)
    for j in range(24378):
        axs[i].plot(t, erfs_t0[age][j], color="silver", alpha=0.1, linewidth=0.1, zorder=2)
    pltkit.StylizeAxes(
        axs[i], 
        facecolor="white",
        title=age_groups[i],
        title_kwargs={"fontsize": 12, "fontweight": "bold"},
        xlabel="Daily temperature (°C)",
        ylim=(0, 16),
        xlim=(-20, 40),
        spines={"top":False, "right":False}
    )
axs[0].set_ylabel("Relative mortality \n [deaths per 100,000 people]", fontsize=12)
plt.tight_layout()

plt.savefig(os.path.dirname(wdir) +"/figures/Paper1/Carleton_ERFs.png", dpi=300, bbox_inches='tight')
plt.show()

### SPARCCLE

#### Time series

In [ ]:
wdir = "X:\\user\\liprandicn/Projects\\mt-comparison\\models/Carleton2022/output/SPARCCLE/"

file_list = sorted(glob.glob(wdir+"*.nc"))

region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

for i in range(len(file_list)):
    filename=re.search(r'SPARCCLE\\(.*?).nc', file_list[i]).group(1)
    scenario_name = re.search(r'SPARCCLE_(.*?)_2000', filename).group(1)
    ds = pltkit.LoadMortality(wdir, filename, region_type, region, t_type, cause, age_group, variable)
    plt.plot(ds.year, ds.values, label=scenario_name, linewidth=1)

main= pltkit.LoadMortality(wdir, "mortality_SPARCCLE_SSP2_Hist_ERA5_NoAdap_2000-2025_mean", region_type, region, t_type, cause, age_group, variable)
plt.plot(main.year, main.values, label="Historical", linewidth=0.6, c="k")

# plt.legend()
plt.show()

In [ ]:
import plotly.graph_objects as go

wdir = "X:\\user\\liprandicn/Projects\\mt-comparison\\models/Carleton2022/output/SPARCCLE/"
file_list = sorted(glob.glob(wdir + "*.nc"))

region_type = "IMAGE"
t_type = "heat"
variable = "mortality"
age_group = "All ages"
region = "World"
cause = None

fig = go.Figure()

for i in range(len(file_list)):
    filename = re.search(r'SPARCCLE\\(.*?).nc', file_list[i]).group(1)
    scenario_name = re.search(r'SPARCCLE_(.*?)_2000', filename).group(1)
    ds = pltkit.LoadMortality(wdir, filename, region_type, region, t_type, cause, age_group, variable)#.rolling(year=30, center=True).mean(dim="time")
    
    fig.add_trace(go.Scatter(
        x=ds.year, 
        y=ds.values, 
        mode='lines',
        name=scenario_name,
        line=dict(width=2),
        hovertemplate=f"<b>Scenario:</b> {scenario_name}<br>" +
                      "Year: %{x}<br>" +
                      "Value: %{y}<extra></extra>"
    ))

fig.update_layout(
    xaxis_title="Año",
    yaxis_title=variable.capitalize(),
    hovermode="closest",
    width=1200,
    height=1000
)

fig.show()

#### IMPACTS Covariates

#### GMST

In [ ]:
wdir = "X:/user/dekkerm/IMAGE_environments/IMPACTS/Z_Emulator_Standalone_Tool/"

scenarios = ["SSP1_M_CP_ERA_NoEcon", "SSP1_M_CP_ERA_AllImpacts", "SSP3_H_ERA_AllImpacts"]
var = "GTMP_30MIN"

for scenario in scenarios:

    SSP_temp = xr.open_dataset(wdir +f"{scenario}/netcdf/{var}.NC")
    SSP_temp.mean(dim="latitude").mean(dim="longitude").mean(dim="NM")[var].plot(label=scenario)

plt.legend()
plt.title("Global mean temperature projections (unweighted)")
plt.show()

#### Income

In [ ]:
from dataclasses import dataclass
@dataclass
class Sets:
    gdp_dir: str
    scenario: str
    wdir: str


wdir = "X:\\user\\liprandicn\\projects/mt-comparison\\models/carleton2022\\data\\CarletonSM\\econ_vars\\"
# G_SSP1 = xr.open_dataset(wdir+"SSP1.nc4")
# G_SSP2 = xr.open_dataset(wdir+"SSP2.nc4")
# G_SSP3 = xr.open_dataset(wdir+"SSP3.nc4")
# G_SSP5 = xr.open_dataset(wdir+"SSP5.nc4")
# ( G_SSP1.mean(dim="model").sum(dim="region").gdp / G_SSP1.mean(dim="model").sum(dim="region").pop ).plot(label="SSP1")
# ( G_SSP2.mean(dim="model").sum(dim="region").gdp / G_SSP2.mean(dim="model").sum(dim="region").pop ).plot(label="SSP2")
# ( G_SSP3.mean(dim="model").sum(dim="region").gdp / G_SSP3.mean(dim="model").sum(dim="region").pop ).plot(label="SSP3")
# ( G_SSP5.mean(dim="model").sum(dim="region").gdp / G_SSP5.mean(dim="model").sum(dim="region").pop ).plot(label="SSP5")

# wdir = "X:\\user\\liprandicn\\Data\\IMAGE_temperature\\IMP-SSP1-REF-GDPIMP-15\\GDPpc_incl_impacts.out"
# sets = Sets(gdp_dir=wdir, scenario="SSP1")

# GDPPC_VLLO = mf.ReadTIMERFiles(sets)
# GDPPC_VLLO.sel(region="World").Value.plot(label="SSP1_impacts", color="k", linewidth=0.5)


gdp_dir = "X:/user/dekkerm/IMAGE_environments/IMPACTS/2_TIMER/outputlib/TIMER_3_5/IMPACTS/SSP2_VLLO_STS1_AllImpacts/indicators/Economy/GDPpc_incl_impacts.out"
SSP1_L = mf.ReadTIMERFiles(Sets(gdp_dir=gdp_dir, scenario="SSP1", wdir=None), save=False)
SSP1_L.sel(region="World").Value.plot(label="SSP1_L", color="k", linewidth=2)

plt.legend()
plt.title("Global mean GDPpc PPP projections")
plt.show()

##### GDP and POP trajectories of a region

In [ ]:
region_class = pd.read_csv(f"{wdir}/data/regions/region_classification.csv")

region = "NAF"
regs = region_class[region_class["IMAGE26"].str.contains(region, na=False)]["hierid"]

ssp1 = xr.open_dataset(wdir+f"data/carleton_sm/econ_vars/SSP1.nc4")   
ssp2 = xr.open_dataset(wdir+f"data/carleton_sm/econ_vars/SSP2.nc4")   
ssp3 = xr.open_dataset(wdir+f"data/carleton_sm/econ_vars/SSP3.nc4")   
ssp5 = xr.open_dataset(wdir+f"data/carleton_sm/econ_vars/SSP5.nc4") 

ssp1.sel(region=regs.values).mean(dim="model").sum(dim="region").pop.plot(label="SSP1")
ssp2.sel(region=regs.values).mean(dim="model").sum(dim="region").pop.plot(label="SSP2")
ssp3.sel(region=regs.values).mean(dim="model").sum(dim="region").pop.plot(label="SSP3")
ssp5.sel(region=regs.values).mean(dim="model").sum(dim="region").pop.plot(label="SSP5")
plt.legend()
plt.title(f"Population | {region}")
plt.show()

#### Population

In [ ]:
(G_SSP1.mean(dim="model").sum(dim="region").pop ).plot(label="SSP1_M")
(G_SSP2.mean(dim="model").sum(dim="region").pop ).plot(label="SSP2_M")
( G_SSP3.mean(dim="model").sum(dim="region").pop ).plot(label="SSP3_H")
(  G_SSP5.mean(dim="model").sum(dim="region").pop ).plot(label="SSP5_H")
plt.legend()
plt.title("Global population projections")
plt.show()

### Regression with climate indices

In [ ]:
era5dir = "X:/user/liprandicn/Data/ERA5_indices/"

IOD = xr.open_dataset(era5dir+"IOD_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
NAO = xr.open_dataset(era5dir+"winter_NAO_ERA5_historical_r1i1p1f1_gn_1940-202512.nc")

wdir = "X:/user/liprandicn/mt-comparison/"
regions = ["World", "RSAM", "WEU", "SEAS", "INDIA", "INDO", "OCE"]

def DetrendedMortality(region):
    mortality = pltkit.LoadMortality(wdir, "carleton2022/output/DEFAULT/mortality_default_SSP2_ERA5_2000-2025", range(2000,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial

niño = N34.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values - 273.15
dipole = IOD.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values
oscillation = NAO.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).msl.values

indices = {"Niño": niño, "NAO": oscillation, "IOD": dipole}

fig, ax = plt.subplots(7,3, figsize=(12,20))

for i,region in enumerate(regions):    
    for j, index in enumerate(indices.keys()):
    
        ax[i,j].scatter(indices[index], DetrendedMortality(region), color="navy")    
        ax[i,j].plot(indices[index], np.poly1d(np.polyfit(indices[index], DetrendedMortality(region), 1))(indices[index]), color="navy", alpha=0.5)    
        ax[i,j].text(0.05, 0.95, f"r={np.corrcoef(indices[index], DetrendedMortality(region))[0,1]:.2f}", transform=ax[i,j].transAxes, fontsize=10, 
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax[0,j].set_title(index, fontsize=12)
        
    ax[i,0].set_ylabel(region)
    
plt.show()

In [ ]:
IMAGE = {
    "World": "World",
    "CAN":"Canada", "USA":"USA", "MEX":"Mexico", "RCAM":"Central America", "BRA":"Brazil", "RSAM":"Rest of South America", # America
    "NAF":"Northern Africa", "WAF":"Western Africa", "EAF":"Eastern Africa", "SAF":"South Africa", # Africa
    "WEU":'Western Europe', "CEU":"Central Europe", "TUR":"Turkey", "UKR":"Ukraine", # Europe
    "STAN":"Central Asia", "RUS":"Russia region", "ME":"Middle East", "INDIA":"India", "KOR":"Korea region", "CHN":"China region", "SEAS":"Southeastern Asia", "INDO":"Indonesia region", "JAP":"Japan", # Asia
    "OCE":"Oceania", "RSAS":"Rest of South Asia", "RSAF":"Rest of Southern Africa" # Oceania + other
}

years = range(1970, 2026)
era5dir = "X:/user/liprandicn/Data/ERA5_indices/"
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
wdir = "X:/user/liprandicn/mt-comparison/"
niño = N34.mean("lat").mean("lon").sel(valid_time=slice(str(years[0]), str(years[-1]))).sst.values - 273.15


def DetrendedMortality(region):
    mortality = pltkit.LoadMortality(wdir, "carleton2022/output/modes/IMAGE/MOR_modes_SSP2_ERA5_NoAdap_1970-2025", range(1970,2026), region, "heat", "relative", "all", None, None, None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial


fig, ax = plt.subplots(5,6, figsize=(25,15))
axs = ax.flatten()
colors = plt.cm.viridis(np.linspace(0, 1, len(years)))  # Viridis colormap

for i,region in enumerate(IMAGE.keys()):    
    
    axs[i].scatter(niño, DetrendedMortality(region), c=[colors[year - 1970] for year in years])    
    axs[i].plot(niño, np.poly1d(np.polyfit(niño, DetrendedMortality(region), 1))(niño), alpha=0.5)    
    axs[i].text(0.05, 0.95, f"r={np.corrcoef(niño, DetrendedMortality(region))[0,1]:.2f}", transform=axs[i].transAxes, fontsize=10, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axs[i].set_title(IMAGE[region], fontsize=12)
    if i%6 == 0:
        axs[i].set_ylabel("Deaths per 100,000 people")
    if i>=21:
        axs[i].set_xlabel("N3.4 SST anomaly (°C)")

plt.tight_layout()
plt.show()

### Testing new changes

In [ ]:
temp_type = "heat"
unit = "total"
age_group = "All ages"
region = "World"
cause = None
rt = "IMAGE"
var = "mortality"

wdir1 = "X:/user/liprandicn/projects/mt-comparison/models/carleton2022/output/"

main = pltkit.LoadMortality(wdir1, "SPARCCLE_1stRound/mortality_SPARCCLE_SSP1_M_CP_ERA_NoEcon_2000-2100_mean", rt, region, temp_type, cause, age_group, var)
plt.plot(main.year, main.values, label="SPARCCLE_1stRound", linewidth=2, c="grey")

main= pltkit.LoadMortality(wdir1, "SPARCCLE/mortality_SPARCCLE_SSP1_M_CP_ERA_NoEcon_2050-2052_mean", rt, region, temp_type, cause, age_group, var)
plt.plot(main.year, main.values, label="test", linewidth=1, c="magenta")

plt.title(f"Global mortality {age_group}")
plt.legend()

### Temperature variability

In [ ]:
var_dir = "X:/user/scherrenbm/Data/Internal_Variability_EERIE/"
file_list = sorted(glob.glob(var_dir+"//internal_variability_grid_tas_Amon_ACCESS-CM2_hist*.nc"))

fig, ax = plt.subplots(3,3, figsize=(15,15))
ax=ax.flatten()

for i in range(len(file_list)):
    filename = re.search(r'ACCESS-CM2_hist-(.*?)_gn', file_list[i]).group(1)
    var = xr.open_dataset(file_list[i], decode_times=False).mean(dim="lat").mean(dim="lon").isel(time=slice(-1200, None))
    ax[i].plot(var.IV_tas.time, var.IV_tas.values, label=filename, c=f"C{i}")
    ax[i].legend()
    ax[i].set_ylim(-2,3)
plt.suptitle("ACCESS-CM2 - Months after 2000", y=0.91)
plt.show()

### Shapley

##### LHS

In [ ]:
from scipy.special import erfinv

wdir = "X:/user/liprandicn/projects/mt-comparison/models/carleton2022"

with open(wdir+"/data/CarletonSM/Agespec_interaction_response.csvv") as f:
        
    # Initialize nupt array of 36x36
    sigma_original = np.zeros((36,36))
    
    # Extract relevant lines
    for i, line in enumerate(f, start=1):
            
        if i == 23:
            # Extract gamma coefficients
            gammas = np.array([float(x) for x in line.strip().split(", ")])
            
        if i in range(25,61):
            sigma_original[i-25] = np.array([float(x) for x in line.strip().split(", ")])

N = sigma_original.shape[0]

# Cholesky decomposition
L = np.linalg.cholesky(sigma_original)
# plt.imshow(L, vmax=0.1)
# plt.colorbar()
# plt.title("Cholesky decomposition")

sample_range = range(10, 1000, 10)  
mean_errors = []
iterations_per_size = 50  # Simulations per size to stabilize LHS randomness


# ------------------------- SIMULATION LOOP -------------------------------------------
for M in sample_range:
    iteration_errors = []
    
    for _ in range(iterations_per_size):
        # Initialize Latin Hypercube Sampler
        sampler = qmc.LatinHypercube(d=N)
        uniform_samples = sampler.random(n=M)

        # Inverse transform sampling to standard Normal Distribution N(0,1)
        normal_samples = np.sqrt(2) * erfinv(2 * uniform_samples - 1)

        # Induce the target covariance structure using the Cholesky matrix
        correlated_samples = np.dot(normal_samples, L.T)

        # Compute the empirical covariance matrix from the simulated samples
        sigma_simulated = np.cov(correlated_samples, rowvar=False)

        # Calculate the Frobenius Norm of the matrix difference (relative error metric)
        norma_base = np.linalg.norm(sigma_original, ord='fro')
        frobenius_error = np.linalg.norm(sigma_original - sigma_simulated, ord='fro') / norma_base
        iteration_errors.append(frobenius_error)

    mean_errors.append(np.mean(iteration_errors))

# ------ ELBOW METHOD OPTIMIZATION (inflection point when the relative error is below 2.5%) ---------
relative_changes = np.abs(np.diff(mean_errors) / mean_errors[:-1])
optimal_m = sample_range[-1]  # Default fallback to max samples

for idx, change in enumerate(relative_changes):
    if change < 0.025:  # 2.5% stabilization threshold
        optimal_m = sample_range[idx + 1]
        break

# ---------------------------- CONVERGENCE PLOT -----------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(sample_range, mean_errors, marker='o', color='darkgrey', linewidth=2, label='Frobenius Norm Error')
plt.axvline(x=optimal_m, color='olive', linestyle='--', linewidth=2, label=f'Suggested Minimum: {optimal_m} samples')
plt.title('Covariance Matrix Convergence Curve (LHS + Cholesky)')
plt.xlabel('Number of Samples (M)')
plt.ylabel('Fit Error Magnitude')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
wdir = "X:\\user\\liprandicn/Projects\\mt-comparison\\models/Carleton2022/output/SPARCCLE_uncertainty\\"

file_list = sorted(glob.glob(wdir+"*.nc"))

region_type="IMAGE"
t_type = "heat"
variable="mortality"
age_group = "All ages"
region = "World"
years = range(2000,2101)
cause=None

for i in range(len(file_list)):
    filename = re.search(r'([^\\/]+)\.nc$', file_list[i]).group(1)
    scenario_name = re.search(r'uncertainty_(.*?)_2000', filename).group(1)
    ds = pltkit.LoadMortality(wdir, filename, region_type, region, t_type, cause, age_group, variable)
    plt.plot(ds.year, ds.values, label=scenario_name, linewidth=0.2, c="C0")
    
wdir1 = "X:\\user\\liprandicn/Projects\\mt-comparison\\models/Carleton2022/output/SPARCCLE/"
main= pltkit.LoadMortality(wdir1, "mortality_SPARCCLE_SSP2_Hist_ERA5_NoAdap_2000-2025_mean", region_type, region, t_type, cause, age_group, variable)
plt.plot(main.year, main.values, label="Historical", linewidth=0.5, c="k")

# plt.legend()
plt.show()